Process sessions
Jira: https://keyless.atlassian.net/browse/BIOM-625

In [ ]:
import shutil

import numpy as np
import pandas as pd
from loguru import logger

from distutils.dir_util import copy_tree
from pathlib import Path

import cv2
import datasets
import os
import s3fs

import modules.globals
from modules import core
from modules.face_analyser import get_one_face

from tqdm.auto import tqdm

In [ ]:
ds_root = "s3://sagemaker-production-eu-central-1-kl-biometric-datasets/raw_datasets/face_biometrics/deepfakes/hackathon_2025-07_deepfakes/to_process_to_create_dest_dataset/"
ds_key = "train_aggregation_ensemble/part_0"
output_ds_root = "s3://sagemaker-production-eu-central-1-kl-biometric-datasets/raw_datasets/face_biometrics/deepfakes/hackathon_2025-07_deepfakes/destination_datasets/"

local_original_imgs_root = "/home/sagemaker-user/face_swap/original_images/"
local_swapped_imgs_root = "/home/sagemaker-user/face_swap/swapped_images_enh/"
execution_provider = "cpu"  # cuda or cpu
face_enhancer = False
rng_seed = 42

In [ ]:
### init ###
modules.globals.execution_providers = core.decode_execution_providers(
    [execution_provider]
)
frame_processors = ["face_swapper"]
modules.globals.fp_ui["face_enhancer"] = False
if face_enhancer:
    frame_processors.append("face_enhancer")
    modules.globals.fp_ui["face_enhancer"] = True
np.random.seed(rng_seed)

modules.globals.max_memory = core.suggest_max_memory()
modules.globals.execution_threads = core.suggest_execution_threads()
core.limit_resources()

In [ ]:
ds_path = os.path.join(ds_root, ds_key, "hf_dataset")
ds = datasets.Dataset.load_from_disk(ds_path)

In [ ]:
local_swapped_imgs_path = os.path.join(local_swapped_imgs_root, ds_key)
os.makedirs(local_swapped_imgs_path, exist_ok=True)


In [ ]:
session_img_paths = {}
for row in tqdm(ds):
    session_folder = row["session_folder"]
    session_local_path = os.path.join(local_swapped_imgs_path, session_folder)

    if session_folder not in session_img_paths:
        session_img_paths[session_folder] = []
        os.makedirs(session_local_path, exist_ok=True)
        source_img_path = os.path.join(session_local_path, "source_img.jpg")
        row["source_img_raw"].save(source_img_path)

    img_path = os.path.join(session_local_path, row["photo_name"])
    if not img_path.endswith == ".jpg":
        img_path += ".jpg"
    row["img_raw"].save(img_path)

    session_img_paths[session_folder].append(img_path)

In [ ]:
for session_folder in session_img_paths.keys():
    logger.info(f"Processing session {session_folder}")
    source_img_path = os.path.join(
        os.path.dirname(session_img_paths[session_folder][0]), "source_img.jpg"
    )

    for frame_processor in core.get_frame_processors_modules(frame_processors):
        logger.info(f"Progressing... {frame_processor.NAME}")
        print(f"Total frames: {len(session_img_paths[session_folder])}")
        frame_processor.process_video(
            source_img_path, session_img_paths[session_folder]
        )
        core.release_resources()